# SaaS 30일 이탈 EDA 준비 데이터셋 만들기

## 우리가 지금 뭘 하려는가

지금은 복잡한 score를 보려는 단계가 아니다.
현재 고객 상태를 보고, 이 고객이 앞으로 30일 안에 떠날지 보는 가장 단순한 분석을 하려는 단계다.

그래서 이 notebook의 목적은 딱 하나다.

- `02` 에서 만든 기본 테이블에서
- 지금 질문에 필요한 컬럼만 남겨
- EDA에서 바로 읽을 수 있는 최소 데이터셋을 만드는 것

## 우리가 보고 싶은 질문

1. `monthly` 와 `annual` 중 누가 더 많이 떠나는가
2. `plan tier` 에 따라 이탈이 다른가
3. `days_to_expiry`, `renewal_failure`, `usage_drop` 가 실제 이탈과 연결되는가

그래서 여기서는 이 질문에 필요한 컬럼만 남긴다.

## 원본에서 무엇을 가져왔는가

`02` 에서 이미 아래 현재 상태 컬럼과 라벨을 만들었다.

- segment
  - `current_plan_tier`
  - `current_billing_frequency`
  - `current_auto_renew_flag`
  - `current_mrr_amount`
  - `current_seats`
- current-state feature
  - `days_since_join`
  - `days_to_expiry`
  - `cancel_flag_30d`
  - `renewal_failure_count_90d`
  - `plan_change_up_flag_90d`
  - `plan_change_down_flag_90d`
  - `usage_intensity_30d`
  - `usage_drop_ratio_30d_vs_prev_30d`
- label
  - `leave_next_30d`
  - `leave_reason`

즉 `03` 에서는 이제 원본 raw나 helper를 다시 볼 필요가 없다.
필요한 것만 남긴 최소 분석용 테이블로 정리하면 된다.

## 그래서 무엇을 남길 것인가

최종 `eda_ready` 는 아래만 남긴다.

- 시간축
  - `as_of_date`
- segment
  - `current_plan_tier`, `current_billing_frequency`, `current_auto_renew_flag`, `current_mrr_amount`, `current_seats`
- current-state feature
  - `days_since_join`, `days_to_expiry`, `cancel_flag_30d`, `renewal_failure_count_90d`, `plan_change_up_flag_90d`, `plan_change_down_flag_90d`, `usage_intensity_30d`, `usage_drop_ratio_30d_vs_prev_30d`
- label
  - `leave_next_30d`, `leave_reason`

식별자, 추적용 key, helper, 중복 컬럼은 전부 뺀다.
왜냐하면 지금 목적은 디버그가 아니라 메인 EDA이기 때문이다.

## 진행 순서

- Part 1. `02` 결과 읽기
- Part 2. 현재 컬럼 inventory 보기
- Part 3. `eda_ready` 만들기
- Part 4. 저장 전 sanity check
- Part 5. 저장하기


In [ ]:
from pathlib import Path

import pandas as pd

CWD = Path.cwd().resolve()
WORKSPACE_ROOT = next(
    (path for path in [CWD, *CWD.parents] if (path / 'back_research').exists() and (path / 'docs').exists()),
    CWD,
)
NOTEBOOK_ROOT = WORKSPACE_ROOT / 'back_research' / 'wonbeenlee' / 'notebooks' / 'branch_workspaces' / 'membership_subscription_lifecycle'
OUTPUT_DIR = NOTEBOOK_ROOT / 'eda' / 'outputs'
EDA_READY_DIR = OUTPUT_DIR / 'eda_ready'
EDA_READY_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CSV = OUTPUT_DIR / 'saas_leave_next_30d_modeling_base.csv'
MODEL_PARQUET = OUTPUT_DIR / 'saas_leave_next_30d_modeling_base.parquet'

print({'model_csv': str(MODEL_CSV), 'exists': MODEL_CSV.exists()})

## Part 1. `02` 결과 읽기

여기서는 `saas_leave_next_30d_modeling_base` 한 파일만 읽는다.
이미 필요한 현재 상태 컬럼과 라벨이 다 들어 있는 기본 테이블이다.

In [ ]:
def load_prefer_csv(csv_path: Path, parquet_path: Path) -> pd.DataFrame:
    if csv_path.exists():
        return pd.read_csv(csv_path, low_memory=False)
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    raise FileNotFoundError(f'Could not find export file: {csv_path.name} or {parquet_path.name}. Run notebook 02 first.')

model_base = load_prefer_csv(MODEL_CSV, MODEL_PARQUET)
print({'model_base_shape': model_base.shape})
model_base.head()

## Part 2. 현재 컬럼 inventory 보기

이 part에서는 실제로 어떤 컬럼이 들어있는지, null이 얼마나 있는지 간단히 확인한다.
목적은 복잡한 dictionary를 만드는 게 아니라, EDA 전에 구조가 맞는지 확인하는 것이다.

In [ ]:
inventory = pd.DataFrame([
    {
        'column_name': col,
        'dtype': str(model_base[col].dtype),
        'null_ratio': round(float(model_base[col].isna().mean()), 4),
        'sample_value': model_base[col].dropna().iloc[0] if model_base[col].dropna().shape[0] > 0 else None,
    }
    for col in model_base.columns
]).sort_values(['null_ratio', 'column_name'], ascending=[False, True]).reset_index(drop=True)
inventory

## Part 3. `eda_ready` 만들기

지금 기본 테이블도 꽤 단순하지만, 메인 EDA 목적에 맞게 한 번 더 정리한다.
남길 컬럼은 아래뿐이다.

- `as_of_date`
- `current_plan_tier`, `current_billing_frequency`, `current_auto_renew_flag`, `current_mrr_amount`, `current_seats`
- `days_since_join`, `days_to_expiry`, `cancel_flag_30d`, `renewal_failure_count_90d`, `plan_change_up_flag_90d`, `plan_change_down_flag_90d`, `usage_intensity_30d`, `usage_drop_ratio_30d_vs_prev_30d`
- `leave_next_30d`, `leave_reason`


In [ ]:
EDA_COLUMNS = [
    'as_of_date',
    'current_plan_tier',
    'current_billing_frequency',
    'current_auto_renew_flag',
    'current_mrr_amount',
    'current_seats',
    'days_since_join',
    'days_to_expiry',
    'cancel_flag_30d',
    'renewal_failure_count_90d',
    'plan_change_up_flag_90d',
    'plan_change_down_flag_90d',
    'usage_intensity_30d',
    'usage_drop_ratio_30d_vs_prev_30d',
    'leave_next_30d',
    'leave_reason',
]

eda_ready = model_base[EDA_COLUMNS].copy()
print({'eda_ready_shape': eda_ready.shape})
eda_ready.head()

## Part 4. 저장 전 sanity check

저장 전에 아래만 본다.

- leave 비율이 너무 이상하지 않은지
- monthly / annual / plan tier 값이 실제로 있는지
- null이 과도한 컬럼이 있는지


In [ ]:
sanity = {
    'row_count': int(len(eda_ready)),
    'leave_rate': round(float(eda_ready['leave_next_30d'].mean()), 4),
    'billing_values': sorted(eda_ready['current_billing_frequency'].dropna().astype(str).unique().tolist()),
    'plan_values': sorted(eda_ready['current_plan_tier'].dropna().astype(str).unique().tolist()),
}
sanity

In [ ]:
null_summary = pd.DataFrame([
    {
        'column_name': col,
        'null_ratio': round(float(eda_ready[col].isna().mean()), 4),
        'sample_value': eda_ready[col].dropna().iloc[0] if eda_ready[col].dropna().shape[0] > 0 else None,
    }
    for col in eda_ready.columns
]).sort_values(['null_ratio', 'column_name'], ascending=[False, True]).reset_index(drop=True)
null_summary

## Part 5. EDA 데이터셋 저장

최종 출력은 아래다.

- `saas_leave_next_30d_eda_ready.csv`
- `saas_leave_next_30d_eda_ready.parquet` 가능하면 같이 저장


In [ ]:
def export_frame(frame: pd.DataFrame, stem: str, output_dir: Path) -> dict[str, str]:
    csv_path = output_dir / f'{stem}.csv'
    frame.to_csv(csv_path, index=False)

    parquet_path = output_dir / f'{stem}.parquet'
    parquet_result = 'saved'
    try:
        frame.to_parquet(parquet_path, index=False)
    except Exception as exc:
        parquet_result = f'not saved ({type(exc).__name__}: {exc})'

    return {
        'csv_path': str(csv_path),
        'parquet_path': str(parquet_path),
        'parquet_result': parquet_result,
        'row_count': len(frame),
        'column_count': len(frame.columns),
    }

export_result = export_frame(eda_ready, 'saas_leave_next_30d_eda_ready', EDA_READY_DIR)
pd.DataFrame([export_result])